We start with filtering out the non-tubingensis species from our data

In [ ]:
bcftools view \ 
    -S ^"/home/puntm/my_scratch_dir/2026_black_aspergillus/data/no_atubingensis.list" \
    -O z \
    -o /home/puntm/my_scratch_dir/2026_refbased/filtered_species.vcf.gz \
    /home/puntm/my_scratch_dir/2026_refbased/all_scaffolds.filteredSNPs.vcf.gz

In [ ]:
plink \
  --allow-extra-chr \
  --make-bed \
  --set-missing-var-ids @:# \
  --out /home/puntm/my_scratch_dir/2026_refbased/plink/fspecies \
  --indep-pairwise 50 10 0.1 \
  --geno 0.1 \
  --maf 0.01 \
  --snps-only just-acgt \
  --vcf /home/puntm/my_scratch_dir/2026_refbased/filtered_species.vcf.gz \
  --vcf-half-call missing
#recode the : to tab
sed 's/:/\t/g' /home/puntm/my_scratch_dir/2026_refbased/plink/fspecies.prune.in > /home/puntm/my_scratch_dir/2026_refbased/plink/targets.txt

bcftools view -T /home/puntm/my_scratch_dir/2026_refbased/plink/targets.txt /home/puntm/my_scratch_dir/2026_refbased/filtered_species.vcf.gz --output /home/puntm/my_scratch_dir/2026_refbased/pruned.vcf.gz

/home/puntm/my_scratch_dir/local_software/vcf2phylip.py --input /home/puntm/my_scratch_dir/2026_refbased/pruned.vcf.gz -f 

Ok, now we have pruned the data for only the atubingensis species, now I will build a new tree to enable new PCA covariates to use as input in plink


In [ ]:
fasttree -gtr -nt < /home/puntm/my_scratch_dir/2026_refbased/pruned.min4.fasta > /home/puntm/my_scratch_dir/2026_black_aspergillus//data/tree_data/fspecies.tree

In [3]:
import pandas as pd
import numpy as np
import os

# Load the data

input_path_1 = '../data/2026_MIC_combined.csv'
df = pd.read_csv(input_path_1)


# --- Configuration ---
strain_col = 'isolate'
mic_cols = ['itc', 'pos', 'vrc'] 
# ---------------------

# 1. Initialize the master dataframe with PLINK ID columns
final_pheno = pd.DataFrame()
final_pheno['FID'] = df[strain_col]
final_pheno['IID'] = df[strain_col]


# 2. Loop through each drug and add the Log2 column
for drug in mic_cols:
    # Convert to numeric (handles 'NA' strings)
    numeric_mic = pd.to_numeric(df[drug], errors='coerce')
    
    # Calculate Log2 (using a mask to avoid log(0) errors if any MICs are 0)
    final_pheno[f'Log2_{drug}'] = np.log2(numeric_mic.replace(0, np.nan))

# 3. Handle Missing Data
# Replacing NaNs with -9 (Standard PLINK missing value indicator)
#final_pheno = final_pheno.fillna(int(0))
#final_pheno['FID'] = int(0)

# 4. Save to one single file
output_path = '../data/GWAS/all_mic_phenotypes.txt'

# Ensure the results directory exists
os.makedirs('../data/GWAS', exist_ok=True)

final_pheno.to_csv(output_path, sep='\t', index=False, quoting=3)

print(f"Successfully created {output_path} with traits: {', '.join(mic_cols)}")
print(final_pheno.head())

Successfully created ../data/GWAS/all_mic_phenotypes.txt with traits: itc, pos, vrc
       FID      IID  Log2_itc  Log2_pos  Log2_vrc
0  V273-64  V273-64       0.0      -2.0       0.0
1  M039-56  M039-56       1.0      -1.0       1.0
2  M039-58  M039-58       0.0       1.0      -2.0
3  V080-81  V080-81       0.0      -2.0       0.0
4  V072-57  V072-57       5.0       1.0      -2.0


The below script converts a nwk file to matrix format, I changed the input tree from the raxml one to a fasttree version that only uses tubingesis sensu stricto

In [20]:
from ete3 import Tree
import pandas as pd

t = Tree("../data/tree_data/fspecies.tree", format=1)
samples = [leaf.name for leaf in t.iter_leaves()]
distances = pd.DataFrame(index=samples, columns=samples)

for i in samples:
    for j in samples:
        try:
            node_i = t.search_nodes(name=i)[0]
            node_j = t.search_nodes(name=j)[0]

            distances.loc[i, j] = t.get_distance(node_i, node_j)
        except IndexError:
            print(f"Missing node: {i} or {j}")
            distances.loc[i, j] = float('nan')

distances.to_csv("../data/GWAS/species_distances.tsv", sep="\t")

Now we convert the distances to a PCA covariates format to be used in plink:

In [21]:
import pandas as pd
from sklearn.decomposition import PCA

dist = pd.read_csv("../data/GWAS/species_distances.tsv", sep="\t", index_col=0)
pca = PCA(n_components=10)
pcs = pca.fit_transform(dist)

# Convert to PLINK covariate format
# PLINK expects: FID, IID, PC1, PC2, ...
pca_df = pd.DataFrame(pcs, columns=[f'PC{i+1}' for i in range(10)])
pca_df.insert(0, 'IID', dist.index)
pca_df.insert(0, 'FID', dist.index)
# Save for PLINK
pca_df.to_csv("../data/GWAS/pca_covariates.txt", sep="\t", index=False)

First I ran the code below to perform PLINK analysis, but from the QQ plot of the P values it became apparent that our lambda is >14 even when excluding the non-tubingensis species. 

In [ ]:
%%bash
VCF_FILE="/home/puntm/my_scratch_dir/2026_refbased/filtered_species.vcf.gz"

plink \
    --vcf "${VCF_FILE}" \
    --allow-extra-chr \
    --all-pheno \
    --linear \
    --allow-no-sex \
    --pheno "../data/GWAS/all_mic_phenotypes.txt" \
    --covar "../data/GWAS/pca_covariates.txt" \
    --out GWAS



Prepare the data for the gemma analysis

In [ ]:
%%bash
VCF_FILE="/home/puntm/my_scratch_dir/2026_refbased/filtered_species.vcf.gz"
out_dir="/home/puntm/my_scratch_dir/2026_black_aspergillus/results/GWAS"

plink \
    --vcf "${VCF_FILE}" \
    --allow-extra-chr \
    --make-bed \
    --out ${out_dir}/gemma_input

awk '{$6=1; print}' ${out_dir}/gemma_input.fam > ${out_dir}/gemma_input.fam.tmp && mv ${out_dir}/gemma_input.fam.tmp ${out_dir}/gemma_input.fam

here I prepare the relatedness test 

In [ ]:
out_dir="/home/puntm/my_scratch_dir/2026_black_aspergillus/results/GWAS"

gemma -bfile ${out_dir}/gemma_input -gk 1 -o fungal_kinship
mv output/fungal_kinship.cXX.txt ${out_dir}/fungal_kinship.cXX.txt
rm -rf output/


The below code prepares the MIC data to fit as gemma input 

In [ ]:
# Define the source file
PHENO_FILE="/home/puntm/my_scratch_dir/2026_black_aspergillus/data/GWAS/all_mic_phenotypes.txt"
out_dir="/home/puntm/my_scratch_dir/2026_black_aspergillus/results/GWAS"
in_dir="/home/puntm/my_scratch_dir/2026_black_aspergillus/data/GWAS"

# Create GEMMA file for Log2_itc (Column 3 of your pheno file)
awk 'NR==FNR {itc[$2]=$3; next} {print (itc[$2]!="" ? itc[$2] : "-9")}' \
    $PHENO_FILE ${out_dir}/gemma_input.fam > ${in_dir}/pheno_itc.txt

# Create GEMMA file for Log2_pos (Column 4 of your pheno file)
awk 'NR==FNR {pos[$2]=$4; next} {print (pos[$2]!="" ? pos[$2] : "-9")}' \
    $PHENO_FILE ${out_dir}/gemma_input.fam > ${in_dir}/pheno_pos.txt

# Create GEMMA file for Log2_vrc (Column 5 of your pheno file)
awk 'NR==FNR {vrc[$2]=$5; next} {print (vrc[$2]!="" ? vrc[$2] : "-9")}' \
    $PHENO_FILE ${out_dir}/gemma_input.fam > ${in_dir}/pheno_vrc.txt

Below the submission to run the analysis for the 3 azoles with gemma

In [ ]:

for i in itc pos vrc; do
    in_dir="/home/puntm/my_scratch_dir/2026_black_aspergillus/data/GWAS"
    out_dir="/home/puntm/my_scratch_dir/2026_black_aspergillus/results/GWAS"
    mkdir -p ${out_dir}/results/GWAS/

    bsub -J "GEMMA_${i}" \
         -n 4 \
         -W 04:00 \
         -M 16G \
         -R "rusage[mem=16G] span[hosts=1]" \
         -o "${out_dir}/gemma_${i}_%J.log" <<EOF
            
            # These commands run on the compute node
            source activate samtools
            
            gemma -bfile gemma_input \
                  -k ${out_dir}fungal_kinship.cXX.txt \
                  -p ${in_dir}/pheno_${i}.txt \
                  -n 1 \
                  -lmm 1 \
                  -o ${out_dir}/GWAS_MIC_col_${i}
EOF
done